<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula_28a_MLP_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1437, 64)
(360, 64)
(1437,)
(360,)


In [13]:
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow import keras
import numpy as np
from sklearn.metrics import accuracy_score

class ShallowNeuralNetwork(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200):
    self.max_iter = max_iter
    self.model = None
  def fit(self, X, y):
    input_shape = X.shape[1]
    output_shape = len(np.unique(y))
    self.model = keras.Sequential([
      keras.layers.Input(shape=(input_shape,)),
      keras.layers.Dense(output_shape, activation='softmax')
    ])
    self.model.compile(optimizer='adam',
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
    self.model.fit(X, y, epochs=self.max_iter, verbose=0)
    return self
  def predict_proba(self, X):
    return self.model.predict(X)
  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return y_pred

clf = ShallowNeuralNetwork()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
0.9666666666666667


In [14]:
clf.model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,952 (7.63 KB)

 Trainable params: 650 (2.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,302 (5.09 KB)

In [15]:
class SigleHiddenLayer(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200, n_hidden_neurons=128):
    self.max_iter = max_iter
    self.n_hidden_neurons = n_hidden_neurons
    self.model = None
  def fit(self, X, y):
    input_shape = X.shape[1]
    output_shape = len(np.unique(y))
    self.model = keras.Sequential([
      keras.layers.Input(shape=(input_shape,)),
      keras.layers.Dense(self.n_hidden_neurons, activation='relu'),
      keras.layers.Dense(output_shape, activation='softmax')
    ])
    self.model.compile(optimizer='adam',
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
    self.model.fit(X, y, epochs=self.max_iter, verbose=0)
    return self
  def predict_proba(self, X):
    return self.model.predict(X)
  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return y_pred

clf = SigleHiddenLayer()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
0.9833333333333333


In [22]:
class MultiLayerPerceptron(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200, n_hidden=[128]):
    self.max_iter = max_iter
    self.n_hidden = n_hidden
    self.model = None
  def fit(self, X, y):
    input_shape = X.shape[1]
    output_shape = len(np.unique(y))

    self.model = keras.Sequential()
    self.model.add(keras.layers.Input(shape=(input_shape,)))
    for n_hidden_neurons in self.n_hidden:
      self.model.add(keras.layers.Dense(n_hidden_neurons, activation='relu'))
    self.model.add(keras.layers.Dense(output_shape, activation='softmax'))

    self.model.compile(optimizer='adam',
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
    self.model.fit(X, y, epochs=self.max_iter, verbose=0)
    return self
  def predict_proba(self, X):
    return self.model.predict(X)
  def predict(self, X):
    y_pred = np.argmax(self.predict_proba(X), axis=1)
    return y_pred

clf = MultiLayerPerceptron(n_hidden=[128,128])
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(accuracy_score(y_test, y_pred))

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
0.9861111111111112
